Develop an interactive tool that allows users to analyze and compare Real vs. Fake news patterns.

In [1]:

%%writefile app.py

import pandas as pd
import streamlit as st
import plotly.express as px

from sklearn.feature_extraction.text import CountVectorizer


st.set_page_config(
    page_title="VeriFact - News Analysis",
    layout="wide"
)

# Dashboard styling
st.markdown("""
<style>
.block-container {
    max-width: 1400px;
    padding-top: 2rem;
}
[data-testid="stMetric"] {
    background-color: #172338;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #263954;
}
</style>
""", unsafe_allow_html=True)


# Load datasets
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

# Add category
real["News Category"] = "Real"
fake["News Category"] = "Fake"

# Combine data
data = pd.concat([real, fake], ignore_index=True)

# Prepare news text
data["news_text"] = data["title"].fillna("") + " " + data["text"].fillna("")

data["word_count"] = data["news_text"].str.split().str.len()


# Dashboard heading
st.title("VeriFact – Real vs Fake News Analysis")
st.write("Interactive tool to compare patterns in real and fake news articles.")

st.divider()


# News counts
st.subheader("News Overview")

col1, col2, col3 = st.columns(3)

col1.metric("Total Articles", f"{len(data):,}")
col2.metric("Real News Articles", f"{len(real):,}")
col3.metric("Fake News Articles", f"{len(fake):,}")


st.divider()


# Average article length
st.subheader("Article Length Comparison")

length_data = data.groupby("News Category")["word_count"].mean().reset_index()

fig1 = px.bar(
    length_data,
    x="News Category",
    y="word_count",
    color="News Category",
    title="Average Words per Article",
    labels={
        "word_count": "Average Word Count",
        "News Category": "News Category"
    }
)

fig1.update_layout(
    template="plotly_dark",
    height=400
)

st.plotly_chart(fig1, use_container_width=True)


# Subject comparison
st.subheader("News Subject Comparison")

if "subject" in data.columns:

    subject_data = data.groupby(
        ["subject", "News Category"]
    ).size().reset_index(name="Article Count")

    fig2 = px.bar(
        subject_data,
        x="subject",
        y="Article Count",
        color="News Category",
        barmode="group",
        title="Real vs Fake Articles by Subject"
    )

    fig2.update_layout(
        template="plotly_dark",
        height=450
    )

    st.plotly_chart(fig2, use_container_width=True)


# Common word comparison
st.subheader("Common Words Analysis")

selected_category = st.selectbox(
    "Choose news category",
    ["Real", "Fake"]
)

selected_text = data[data["News Category"] == selected_category]["news_text"]

vectorizer = CountVectorizer(
    stop_words="english",
    max_features=15
)

word_counts = vectorizer.fit_transform(selected_text)

word_frequency = pd.DataFrame({
    "Word": vectorizer.get_feature_names_out(),
    "Frequency": word_counts.sum(axis=0).A1
})

word_frequency = word_frequency.sort_values(
    "Frequency",
    ascending=False
)

fig3 = px.bar(
    word_frequency,
    x="Frequency",
    y="Word",
    orientation="h",
    title=f"Most Common Words in {selected_category} News"
)

fig3.update_layout(
    template="plotly_dark",
    height=500
)

st.plotly_chart(fig3, use_container_width=True)


st.divider()


# Interactive article explorer
st.subheader("Explore Articles")

col1, col2 = st.columns(2)

with col1:
    category_filter = st.selectbox(
        "Select News Category",
        ["All", "Real", "Fake"]
    )

with col2:
    subject_filter = st.selectbox(
        "Select Subject",
        ["All"] + sorted(data["subject"].dropna().unique().tolist())
        if "subject" in data.columns
        else ["All"]
    )


filtered_data = data.copy()

if category_filter != "All":
    filtered_data = filtered_data[
        filtered_data["News Category"] == category_filter
    ]

if subject_filter != "All" and "subject" in data.columns:
    filtered_data = filtered_data[
        filtered_data["subject"] == subject_filter
    ]


st.metric("Matching Articles", f"{len(filtered_data):,}")


# Display headlines
st.dataframe(
    filtered_data[["title", "News Category"] + (["subject"] if "subject" in data.columns else [])],
    use_container_width=True,
    height=400
)


# Read article
st.subheader("Read an Article")

article_titles = filtered_data["title"].dropna().tolist()

if article_titles:

    selected_title = st.selectbox(
        "Choose an article",
        article_titles
    )

    selected_article = filtered_data[
        filtered_data["title"] == selected_title
    ].iloc[0]

    st.markdown("### " + selected_article["title"])

    st.write(selected_article["news_text"])

    st.caption("Category: " + selected_article["News Category"])


Overwriting app.py
